In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import os

In [2]:
spark = (
    SparkSession.builder
    .appName("pagila-tasks")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .config("spark.sql.shuffle.partitions", "8")  # custom
    .getOrCreate()
)

In [3]:
jdbc_url = f"jdbc:postgresql://db:5432/{os.environ['DB_NAME']}"
props = {
    "user": os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
    "driver": "org.postgresql.Driver",
}

In [4]:
def read_table(table, columns=None, where=None):
    """
    move filtering to the postgres level
    """
    cols = ", ".join(columns) if columns else "*"
    query = f"(SELECT {cols} FROM {table}" + (f" WHERE {where}" if where else "") + ") AS t"
    return spark.read.jdbc(url=jdbc_url, table=query, properties=props)

In [5]:
# Output the number of movies in each category, sorted in descending order
film_category = read_table("film_category", ["film_id", "category_id"])
category = read_table("category", ["category_id", "name"])

res1 = (
    film_category.join(F.broadcast(category), 'category_id')
    .groupBy('name')
    .agg(F.count('category_id').alias('movie_count'))
    .orderBy(F.desc("movie_count"))
)
res1.show()

+-----------+-----------+
|       name|movie_count|
+-----------+-----------+
|      Music|        152|
|      Drama|        152|
|     Travel|        151|
|      Games|        150|
|    Foreign|        150|
|   Children|        150|
|     Sci-Fi|        149|
|     Action|        149|
|  Animation|        148|
|     Family|        147|
|   Classics|        147|
|        New|        147|
|Documentary|        145|
|     Sports|        145|
|     Comedy|        143|
|     Horror|        142|
+-----------+-----------+



In [ ]:
res1.explain("formatted")